# Pellet Detection & Classification Master Pipeline

This notebook centralizes all parameters and training steps. You can tweak parameters directly in the cells before running them.

## 1. Prepare YOLO Dataset
Reorganizes the `frames/` output into a YOLO-compatible structure.

In [ ]:
import os
import shutil
import random
from glob import glob

def prepare_yolo_dataset():
    src_dir = 'frames'
    dataset_dir = 'yolo_dataset'
    
    # Create YOLO directory structure
    for split in ['train', 'val']:
        os.makedirs(os.path.join(dataset_dir, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(dataset_dir, 'labels', split), exist_ok=True)
        
    # Get all jpg files that have a corresponding txt file
    all_jpgs = glob(os.path.join(src_dir, '*.jpg'))
    valid_pairs = []
    
    for jpg_path in all_jpgs:
        txt_path = jpg_path.replace('.jpg', '.txt')
        if os.path.exists(txt_path):
            valid_pairs.append((jpg_path, txt_path))
            
    print(f"Found {len(valid_pairs)} valid image-label pairs.")
    
    # Shuffle and split (80% train, 20% val)
    random.shuffle(valid_pairs)
    split_idx = int(len(valid_pairs) * 0.8)
    
    train_pairs = valid_pairs[:split_idx]
    val_pairs = valid_pairs[split_idx:]
    
    def copy_pairs(pairs, split):
        for jpg, txt in pairs:
            basename = os.path.basename(jpg)
            shutil.copy(jpg, os.path.join(dataset_dir, 'images', split, basename))
            shutil.copy(txt, os.path.join(dataset_dir, 'labels', split, basename.replace('.jpg', '.txt')))
            
    copy_pairs(train_pairs, 'train')
    copy_pairs(val_pairs, 'val')
    print(f"Copied {len(train_pairs)} to train and {len(val_pairs)} to val.")
    
    # Create dataset.yaml
    abs_path = os.path.abspath(dataset_dir).replace('\\', '/')
    yaml_content = f"""path: {abs_path}
train: images/train
val: images/val

names:
  0: pellet
"""
    with open('dataset.yaml', 'w') as f:
        f.write(yaml_content)
    print("Created dataset.yaml.")

if __name__ == '__main__':
    prepare_yolo_dataset()


## 2. Train YOLOv11 Base Tracker
Trains the YOLO nano model on the bounding boxes.

In [ ]:
from ultralytics import YOLO

def main():
    print("Loading YOLO11 Nano model...")
    # Load a model
    model = YOLO("yolo11n.pt")

    print("Starting training...")
    # Train the model on our custom dataset
    # We use a small number of epochs (e.g. 5) just for rapid prototyping right now.
    # It can be increased later for full accuracy.
    results = model.train(data="dataset.yaml", epochs=5, imgsz=512, device=0)

if __name__ == '__main__':
    main()


## 3. Extract Sequences for LSTM
Parses the bounding boxes to extract 10-frame tumbling sequences for the deep learning model.

In [ ]:
import os
import csv
from collections import defaultdict
import numpy as np
from PIL import Image

SEQUENCE_LENGTH = 10
CROP_SIZE = 64
IMG_WIDTH = 512
IMG_HEIGHT = 512

def extract_sequences():
    os.makedirs('sequence_dataset/single', exist_ok=True)
    os.makedirs('sequence_dataset/agglomerate', exist_ok=True)
    
    # Read CSV
    pellet_data = defaultdict(list)
    with open('dataset.csv', 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            pellet_data[int(row['pellet_id'])].append({
                'frame': int(row['frame']),
                'is_agglomerate': row['is_agglomerate'] == 'True',
                'x': float(row['x_center']),
                'y': float(row['y_center']),
                'w': float(row['width']),
                'h': float(row['height'])
            })
            
    print(f"Loaded data for {len(pellet_data)} unique pellets.")
    
    seq_count = 0
    for pid, records in pellet_data.items():
        # Sort by frame
        records.sort(key=lambda x: x['frame'])
        
        # We need continuous sequences of length SEQUENCE_LENGTH
        for i in range(len(records) - SEQUENCE_LENGTH + 1):
            seq_records = records[i:i+SEQUENCE_LENGTH]
            
            # Verify they are consecutive frames
            is_consecutive = True
            for j in range(1, SEQUENCE_LENGTH):
                if seq_records[j]['frame'] != seq_records[j-1]['frame'] + 1:
                    is_consecutive = False
                    break
                    
            if not is_consecutive:
                continue
                
            # Extract crops
            seq_tensors = []
            is_agg = seq_records[0]['is_agglomerate']
            
            for r in seq_records:
                img_path = f"frames/frame_{r['frame']}.jpg"
                if not os.path.exists(img_path):
                    break
                    
                img = Image.open(img_path).convert('RGB')
                
                # De-normalize coordinates
                x_center = r['x'] * IMG_WIDTH
                y_center = r['y'] * IMG_HEIGHT
                w = r['w'] * IMG_WIDTH
                h = r['h'] * IMG_HEIGHT
                
                # Crop bounding box (with a little padding to see context)
                padding = 0.2
                w = w * (1 + padding)
                h = h * (1 + padding)
                
                left = int(max(0, x_center - w/2))
                right = int(min(IMG_WIDTH, x_center + w/2))
                top = int(max(0, y_center - h/2))
                bottom = int(min(IMG_HEIGHT, y_center + h/2))
                
                # If bounding box is invalid, skip
                if right <= left or bottom <= top:
                    break
                    
                crop = img.crop((left, top, right, bottom))
                crop = crop.resize((CROP_SIZE, CROP_SIZE))
                
                # Convert to numpy array (H, W, C) -> (C, H, W)
                crop_arr = np.array(crop).transpose(2, 0, 1)
                # Normalize to 0-1
                crop_arr = crop_arr.astype(np.float32) / 255.0
                
                seq_tensors.append(crop_arr)
                
            if len(seq_tensors) == SEQUENCE_LENGTH:
                # Shape: (10, 3, 64, 64)
                seq_np = np.stack(seq_tensors)
                
                folder = 'agglomerate' if is_agg else 'single'
                out_path = os.path.join('sequence_dataset', folder, f'pid_{pid}_seq_{seq_count}.npy')
                np.save(out_path, seq_np)
                seq_count += 1
                
    print(f"Extracted {seq_count} sequences.")

if __name__ == '__main__':
    extract_sequences()


## 4. Train Deep Sequence Classifier
Trains the CNN+LSTM sequence model on the tumbling sequences.

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class PelletSequenceDataset(Dataset):
    def __init__(self, data_dir):
        self.files = []
        self.labels = []
        
        agg_files = glob.glob(os.path.join(data_dir, 'agglomerate', '*.npy'))
        for f in agg_files:
            self.files.append(f)
            self.labels.append(1)
            
        single_files = glob.glob(os.path.join(data_dir, 'single', '*.npy'))
        for f in single_files:
            self.files.append(f)
            self.labels.append(0)
            
    def __len__(self):
        return len(self.files)
        
    def __getitem__(self, idx):
        seq = np.load(self.files[idx])
        # shape (10, 3, 64, 64)
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)

class SequenceClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # Simple CNN feature extractor for 64x64 images
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 32x32
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 16x16
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), # 8x8
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU()
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(64, 2)
        
    def forward(self, x):
        # x is (B, Seq, C, H, W)
        B, Seq, C, H, W = x.shape
        # Flatten batch and seq to run through CNN
        x = x.view(B * Seq, C, H, W)
        features = self.cnn(x)
        # Reshape for LSTM
        features = features.view(B, Seq, -1)
        lstm_out, (hn, cn) = self.lstm(features)
        # Take the output of the last time step
        out = self.fc(lstm_out[:, -1, :])
        return out

def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    dataset = PelletSequenceDataset('sequence_dataset')
    print(f"Loaded {len(dataset)} sequences for training.")
    
    if len(dataset) == 0:
        print("No data found! Generate more simulation data.")
        return
        
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    model = SequenceClassifier().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    for epoch in range(10): # Short epochs for rapid prototyping
        model.train()
        total_loss = 0
        correct = 0
        
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == batch_y).sum().item()
            
        acc = correct / len(dataset)
        print(f"Epoch {epoch+1}/10 - Loss: {total_loss/len(loader):.4f} - Acc: {acc:.4f}")
        
    torch.save(model.state_dict(), 'sequence_model.pt')
    print("Saved model to sequence_model.pt")

if __name__ == '__main__':
    train()


## 5. Master Inference Pipeline
Runs YOLO tracking + ByteTrack + Suspicion filtering + LSTM evaluation in real-time!

In [ ]:
import os
import cv2
import torch
import numpy as np
from PIL import Image
from ultralytics import YOLO
from train_sequence_model import SequenceClassifier

def infer():
    # Load Models
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    import glob
    yolo_model_paths = sorted(glob.glob('runs/detect/train*/weights/best.pt'), key=os.path.getmtime)
    if not yolo_model_paths:
        print("YOLO model not trained yet. Make sure to run the training step first.")
        return
        
    yolo_model_path = yolo_model_paths[-1]
    print(f"Loading YOLO model from: {yolo_model_path}")
        
    yolo_model = YOLO(yolo_model_path)
    
    seq_model_path = 'sequence_model.pt'
    if not os.path.exists(seq_model_path):
        print("Sequence model not trained yet. Make sure to run train_sequence_model.py first.")
        return
        
    seq_model = SequenceClassifier().to(device)
    seq_model.load_state_dict(torch.load(seq_model_path, map_location=device))
    seq_model.eval()
    
    # Memory buffers
    classified = {} # track_id -> "single" or "agglomerate"
    suspicious_buffers = {} # track_id -> list of numpy arrays
    
    os.makedirs('output_frames', exist_ok=True)
    
    # Run tracking on frames
    # Sort the frames numerically
    import glob
    frames = sorted(glob.glob('frames/*.jpg'), key=lambda x: int(os.path.basename(x).split('_')[1].split('.')[0]))
    
    print(f"Running inference on {len(frames)} frames...")
    
    for i, frame_path in enumerate(frames):
        img_bgr = cv2.imread(frame_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        
        # Run YOLO with ByteTrack
        results = yolo_model.track(img_rgb, tracker="bytetrack.yaml", persist=True, verbose=False)
        
        for result in results:
            boxes = result.boxes
            if boxes is None or boxes.id is None:
                continue
                
            for box, track_id_t in zip(boxes, boxes.id):
                track_id = int(track_id_t.item())
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                w = x2 - x1
                h = y2 - y1
                
                # Default label and color
                label = f"ID: {track_id} Pellet"
                color = (255, 0, 0) # Blue (BGR format in OpenCV!)
                
                if track_id in classified:
                    if classified[track_id] == "agglomerate":
                        label = f"ID: {track_id} Agglomerate!"
                        color = (0, 0, 255) # Red
                    else:
                        label = f"ID: {track_id} Single"
                        color = (0, 255, 0) # Green
                else:
                    # Check suspicion
                    aspect_ratio = max(w/h, h/w) if h > 0 and w > 0 else 1.0
                    
                    if aspect_ratio > 1.2:
                        if track_id not in suspicious_buffers:
                            suspicious_buffers[track_id] = []
                            
                        # Crop and process
                        crop = img_rgb[int(y1):int(y2), int(x1):int(x2)]
                        if crop.size > 0:
                            crop_pil = Image.fromarray(crop).resize((64, 64))
                            crop_arr = np.array(crop_pil).transpose(2, 0, 1).astype(np.float32) / 255.0
                            suspicious_buffers[track_id].append(crop_arr)
                            
                            label = f"ID: {track_id} Suspicious ({len(suspicious_buffers[track_id])}/10)"
                            color = (0, 165, 255) # Orange
                            
                            if len(suspicious_buffers[track_id]) == 10:
                                # Classify!
                                seq_tensor = torch.tensor(np.stack(suspicious_buffers[track_id]), dtype=torch.float32).unsqueeze(0).to(device)
                                with torch.no_grad():
                                    outputs = seq_model(seq_tensor)
                                    pred = outputs.argmax(dim=1).item()
                                    
                                if pred == 1:
                                    classified[track_id] = "agglomerate"
                                    label = f"ID: {track_id} Agglomerate!"
                                    color = (0, 0, 255)
                                else:
                                    classified[track_id] = "single"
                                    label = f"ID: {track_id} Single"
                                    color = (0, 255, 0)
                                    
                                # Free up memory
                                del suspicious_buffers[track_id]
                                
                # Draw bounding box
                cv2.rectangle(img_bgr, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                cv2.putText(img_bgr, label, (int(x1), int(y1)-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
                
        # Save output
        out_path = os.path.join('output_frames', os.path.basename(frame_path))
        cv2.imwrite(out_path, img_bgr)
        
    print("Inference complete! Check the output_frames/ folder.")
    
if __name__ == '__main__':
    infer()
